In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# --- Step 1: Load and Combine the Datasets ---
# Update the file paths accordingly.
file_ab = 'C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Datas_Train_Val_Test/full_AB_p_m1_m2 - Sheet1.csv'
file_ba = 'C:/Users/Алишер/Desktop/NazarbayevUniversity/DatasetPdM/Datas_Train_Val_Test/full_BA_p_m1_m2 - Sheet1.csv'

df_ab = pd.read_csv(file_ab)
df_ba = pd.read_csv(file_ba)

# Combine the two datasets
df_full = pd.concat([df_ab, df_ba], ignore_index=True)

# Shuffle to remove any ordering bias
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)



In [2]:
# --- Step 2: Create the Pressure Status Column ---
def pressure_status(p):
    if p < 4.9:
        return 'low'
    elif 4.9 <= p <= 5.2:
        return 'normal'
    else:
        return 'overpressure'

# Create a new column 'pressure_status' based on the continuous 'pressure' column.
df_full['pressure_status'] = df_full['pressure'].apply(pressure_status)



In [4]:
# --- Step 3: Split the Data into Training, Validation, and Test Sets ---
# We use stratification based on 'pressure_status' to keep class distributions similar in each set.

# First, split off the final test set (15% of the data)
df_train_val, df_test = train_test_split(
    df_full, 
    test_size=0.15, 
    random_state=42, 
    stratify=df_full['pressure_status']
)

# Then, split the training+validation set into training and validation sets.
# To have the validation set also be 15% of the original data, we take about 17.65% (15/85) of df_train_val.
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=0.1765, 
    random_state=42, 
    stratify=df_train_val['pressure_status']
)

print("Dataset Sizes:")
print("Total records:", len(df_full))
print("Training set size:", len(df_train))
print("Validation set size:", len(df_val))
print("Test set size:", len(df_test))


Dataset Sizes:
Total records: 846
Training set size: 592
Validation set size: 127
Test set size: 127


In [4]:
print(df_test)

     AC_time  pressure   mech_1  CB_time  pressure  mech_2  BC_time  CA_time  \
578   4140.0        4.0       1   3795.0       4.0       0      NaN      NaN   
646      NaN        NaN       0      NaN       5.0       1   4031.0   2496.0   
346      NaN        NaN       0      NaN       5.0       0   3797.0   2620.0   
628   2605.0        6.0       1   2282.0       6.0       1      NaN      NaN   
276   3798.0        3.5       0   4236.0       3.5       1      NaN      NaN   
..       ...        ...     ...      ...       ...     ...      ...      ...   
2     1854.0        6.0       0   1959.0       6.0       0      NaN      NaN   
620   3590.0        5.0       0   4035.0       5.0       1      NaN      NaN   
810      NaN        NaN       1      NaN       3.5       0   4230.0   3046.0   
319      NaN        NaN       0      NaN       3.5       0   4026.0   2715.0   
508      NaN        NaN       0      NaN       3.5       1   4136.0   2726.0   

     pressure.1 pressure_status  
578  

In [5]:
# --- Step 4: Prepare Features and Targets ---
# Use the four sensor time features. Adjust the feature names if needed.
features = ['AC_time', 'CA_time', 'CB_time', 'BC_time']

# Training features and targets
X_train = df_train[features]
y_train_mech1 = df_train['mech_1']
y_train_mech2 = df_train['mech_2']
y_train_pressure = df_train['pressure_status']

# Validation features and targets
X_val = df_val[features]
y_val_mech1 = df_val['mech_1']
y_val_mech2 = df_val['mech_2']
y_val_pressure = df_val['pressure_status']

# Test features and targets (for later evaluation)
X_test = df_test[features]
y_test_mech1 = df_test['mech_1']
y_test_mech2 = df_test['mech_2']
y_test_pressure = df_test['pressure_status']


In [6]:
# --- Step 5: Train the Classifiers (Using Decision Trees) ---
clf_mech1 = DecisionTreeClassifier(random_state=42)
clf_mech2 = DecisionTreeClassifier(random_state=42)
clf_pressure = DecisionTreeClassifier(random_state=42)

clf_mech1.fit(X_train, y_train_mech1)
clf_mech2.fit(X_train, y_train_mech2)
clf_pressure.fit(X_train, y_train_pressure)

# --- Step 6: Evaluate the Models on the Validation Set ---
print("\nValidation Set Evaluation:")

print("\nClassification Report for mech_1:")
print(classification_report(y_val_mech1, clf_mech1.predict(X_val)))

print("Classification Report for mech_2:")
print(classification_report(y_val_mech2, clf_mech2.predict(X_val)))

print("Classification Report for Pressure Status:")
print(classification_report(y_val_pressure, clf_pressure.predict(X_val)))



Validation Set Evaluation:

Classification Report for mech_1:
              precision    recall  f1-score   support

           0       0.99      0.94      0.96        71
           1       0.93      0.98      0.96        56

    accuracy                           0.96       127
   macro avg       0.96      0.96      0.96       127
weighted avg       0.96      0.96      0.96       127

Classification Report for mech_2:
              precision    recall  f1-score   support

           0       0.87      0.90      0.88        80
           1       0.82      0.77      0.79        47

    accuracy                           0.85       127
   macro avg       0.84      0.83      0.84       127
weighted avg       0.85      0.85      0.85       127

Classification Report for Pressure Status:
              precision    recall  f1-score   support

         low       0.93      0.91      0.92        46
      normal       0.92      0.94      0.93        48
overpressure       1.00      1.00      1.00

In [7]:
# --- Step 7: Predict on the Test Set and Format the Output ---
pred_mech1 = clf_mech1.predict(X_test)
pred_mech2 = clf_mech2.predict(X_test)
pred_pressure = clf_pressure.predict(X_test)

output = {
    "mech_1_prediction": pred_mech1.tolist(),
    "mech_2_prediction": pred_mech2.tolist(),
    "pressure_prediction": pred_pressure.tolist()
}

print("\nTest Set Prediction Output:")
print(output)

# (Optional) Evaluate the predictions on the test set with classification reports.
print("\nTest Set Evaluation:")

print("\nClassification Report for mech_1:")
print(classification_report(y_test_mech1, pred_mech1))

print("Classification Report for mech_2:")
print(classification_report(y_test_mech2, pred_mech2))

print("Classification Report for Pressure Status:")
print(classification_report(y_test_pressure, pred_pressure))



Test Set Prediction Output:
{'mech_1_prediction': [1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0], 'mech_2_prediction': [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0], 'pressure_prediction': ['low', 'normal', 'normal', 'overpressure', 'normal', 'overpressure', 'normal', 'normal', 'low', 'normal', 'overpressure', 'normal', 'low',

In [9]:
import pandas as pd

# -------------------------------
# Function to predict outcomes for new time series input
# -------------------------------
def predict_new_data(new_data_df):
    """
    Given a DataFrame new_data_df with columns:
      'AC_time', 'CA_time', 'CB_time', 'BC_time',
    this function returns a dictionary containing:
      - mech_1_prediction: obstacles on sensor route A->C
      - mech_2_prediction: obstacles on sensor route C->B
      - pressure_prediction: predicted pressure status (low/normal/overpressure)
    """
    # Make predictions using the trained models:
    pred_mech1 = clf_mech1.predict(new_data_df)
    pred_mech2 = clf_mech2.predict(new_data_df)
    pred_pressure = clf_pressure.predict(new_data_df)
    
    # Create an output dictionary:
    output = {
        "mech_1_prediction": pred_mech1.tolist(),
        "mech_2_prediction": pred_mech2.tolist(),
        "pressure_prediction": pred_pressure.tolist()
    }
    
    return output

# -------------------------------
# Example: Manually Pasted Time Series Data
# -------------------------------
# Here, you can paste your new time series data.
# Make sure the columns exactly match those used during training: 
# 'AC_time', 'CA_time', 'CB_time', 'BC_time'

# For example, suppose you have three different time series entries:
new_time_series = pd.DataFrame({
    'AC_time': [3812,3706,3802,3793,3811,4015],   # travel times from sensor A to C
    'CA_time': [2719,2714,2613,2606,2610,2610],   # travel times from sensor C back to A
    'CB_time': [4018,4025,4031,4130,3914,4235],   # travel times from sensor C to B
    'BC_time': [4235,4342,4344,4366,4472,4465]    # travel times from sensor B  to C (4_0_1)
})

# -------------------------------
# Predict and display outputs
# -------------------------------
predictions = predict_new_data(new_time_series)
print("New Data Predictions:")
print(predictions)


New Data Predictions:
{'mech_1_prediction': [0, 1, 0, 0, 0, 0], 'mech_2_prediction': [1, 1, 1, 1, 1, 1], 'pressure_prediction': ['low', 'low', 'low', 'low', 'low', 'low']}
